In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore") # umap deprecation warnings related to numba do not concern us.


# GRN Construction

Builds the prior knowledge network (PKN) from CollecTRI, using the per-phenotype binarized observation data (`meta`) produced by **"4.5. Phenotype Observations.ipynb"**.

`build_pkn` (`src/grn_inference/pkn.py`, rationale and numbers in `src/grn_inference/README.md`) supports two variants, set with `PKN_CORE` below — both on CollecTRI:

- `None` — the rule of the authors' case study 1 code (Chevalier et al. 2025): every TF -> TF edge of CollecTRI, also between TFs absent from the data, plus TF -> target edges for targets present in the data (`MIN_DETERMINED=0`: no check of binary values).
- `"scc"` — data genes plus the largest strongly connected component of the whole CollecTRI (scBoolSeq approach, Magaña-López et al. 2024). The previous version of this notebook is `PKN_CORE="scc"`, `LARGEST_WCC=True`, `DEFAULT_SIGN_UNKNOWN=False`, `UNSOURCE=False`.

`GENE_SET` picks the genes of `meta` the PKN is built on: `"all"` (every gene, as in case study 1) or `"observed+hvg+markers"` — genes determined in at least one phenotype, plus the HVG and markers from "0. Data Preprocessing.ipynb" (`select_genes`). With `INCLUDE_UNMEASURED_TFS=False` the latter gives a PKN of the size of the scBoolSeq retina case study, with most nodes constrained by data.

`INCLUDE_UNMEASURED_TFS=False` keeps only data genes (every node constrained by data). `UNSOURCE=True` adds a positive self-loop to nodes without inputs, as `bonesis.InfluenceGraph.from_sif` does in case study 1.

This notebook ends by exporting `{FILE_PREFIX}_prior_grn.json`, `{FILE_PREFIX}_pkn.json`, `{FILE_PREFIX}_data_meta.json` and `{FILE_PREFIX}_config.json`.

The actual BoNesis solving step lives in **"6. BN Inference (BoNesis).ipynb"**, which consumes these exported files. Set the same `FILE_PREFIX` there.

In [ ]:
import sys
import json
from pathlib import Path as path

import pandas as pd

from markers_retinal import (
    states_and_markers, # Dictionnary containing phenotypes and their marker genes
    marker_genes, # pandas.Series containing simply the name of markers
)

# Repo root → src on path (shared with the RNA velocity / trajectory-inference notebooks)
ROOT = path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from grn_inference import (
    load_collectri,
    build_pkn,
    select_genes,
    in_db,
    df_to_bonesis_data,
    export_pkn_bundle,
)


In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from pipeline_config import load_config

cfg = load_config()  # dataset wybierasz w pipeline_config.toml (`active`) albo env BOOLAI_DATASET
print(cfg.describe())

dataset='hsc_4ds' [single dataset (no 1. Cell Annotation / 2. Data Integration)], samples=['young_A', 'young_B', 'old_A', 'old_B'], method='paga_dpt'


In [ ]:
FILE_PREFIX = cfg.file_prefix  # same bundle prefix as in "4.5. Phenotype Observations.ipynb"
GENE_SET = "observed+hvg+markers"  # "all": every gene of meta | "observed+hvg+markers": genes with >= 1 observation + HVG + markers
PKN_CORE = None  # None: case study 1 rule (all TF->TF + TF->data genes) | "scc": + largest SCC of the whole CollecTRI (scBoolSeq)
MIN_DETERMINED = 0  # data gene = present in meta (0, as in case study 1) or 0/1 in at least this many phenotypes
INCLUDE_UNMEASURED_TFS = False  # False: only data genes in the PKN (True + GENE_SET="all": case study 1)
LARGEST_WCC = False  # keep only the largest weakly connected component
DEFAULT_SIGN_UNKNOWN = True  # CollecTRI "default activation" edges -> sign 0 (solver picks)
UNSOURCE = True  # positive self-loop on nodes without inputs (as InfluenceGraph.from_sif in case study 1)

print(
    f"FILE_PREFIX={FILE_PREFIX!r}, GENE_SET={GENE_SET!r}, PKN_CORE={PKN_CORE!r}, MIN_DETERMINED={MIN_DETERMINED}, "
    f"INCLUDE_UNMEASURED_TFS={INCLUDE_UNMEASURED_TFS}, LARGEST_WCC={LARGEST_WCC}, "
    f"DEFAULT_SIGN_UNKNOWN={DEFAULT_SIGN_UNKNOWN}, UNSOURCE={UNSOURCE}"
)


### TF database loading

In [5]:
net = load_collectri(organism=cfg.organism, cache_path=f"collectri_{cfg.organism}.csv")


In [6]:
meta = pd.read_csv(cfg.meta_path, index_col=0)
with open(f"{FILE_PREFIX}_config.json") as _cfg:
    INCLUDE_HC_CONSTRAINT = json.load(_cfg)["INCLUDE_HC_CONSTRAINT"]

meta_var = meta
print(meta.shape, f"INCLUDE_HC_CONSTRAINT={INCLUDE_HC_CONSTRAINT}")
meta.iloc[:, :10]


(6, 14468) INCLUDE_HC_CONSTRAINT=False


,Xkr4,Mrpl15,Lypla1,Gm37988,Tcea1,Rgs20,Atp6v1h,Rb1cc1,4732440D04Rik,St18
AC,NaN,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,1.0
Cones,NaN,0.0,1.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
NB1,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
NB2,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
RGC,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
RPC,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [7]:
# Observation diagnostics: undetermined (NaN) fraction per phenotype, and genes
# determined in every phenotype with the same value (invariant)
print("NaN fraction per phenotype:")
print(meta.T.isna().mean().round(3).to_string())
invariant = meta.columns[meta.var(skipna=False) == 0]
print(f"\ninvariant genes: {len(invariant)} (in CollecTRI: {in_db(meta[invariant], net).sum()})")


NaN fraction per phenotype:
AC       0.916
Cones    0.901
NB1      0.908
NB2      0.919
RGC      0.897
RPC      0.920

invariant genes: 133 (in CollecTRI: 48)


In [ ]:
# Gene set the PKN is built on. "observed+hvg+markers": genes determined in >= 1 phenotype,
# HVG + markers from "0. Data Preprocessing.ipynb", and the CollecTRI neighbours of markers
# that would otherwise have no edge (they would drop out of the PKN)
if GENE_SET == "all":
    meta_pkn = meta
elif GENE_SET == "observed+hvg+markers":
    _hvg = pd.read_csv(cfg.path("hvgs_csv"), index_col=0, nrows=0).columns
    _markers = pd.Index(marker_genes)
    meta_pkn = meta[select_genes(meta, net, extra_genes=_hvg.union(_markers), connect=_markers)]
else:
    raise ValueError(GENE_SET)
print(f"GENE_SET={GENE_SET!r}: {meta_pkn.shape[1]} / {meta.shape[1]} genes, "
      f"{meta_pkn.notna().any().sum()} with >= 1 observation")


### PKN

In [ ]:
pkn_graph, pkn_report = build_pkn(
    net,
    meta_pkn,
    core=PKN_CORE,
    min_determined=MIN_DETERMINED,
    include_unmeasured_tfs=INCLUDE_UNMEASURED_TFS,
    largest_wcc=LARGEST_WCC,
    default_sign_unknown=DEFAULT_SIGN_UNKNOWN,
)
pkn_report


In [9]:
print(f"markers in PKN: {marker_genes.isin(pkn_graph).mean():.2%}")
print("\t".join(["Cell", "pObserved", "pPKN"]))
_observed = meta.columns[meta.notna().any()]
for pheno, genes in states_and_markers.items():
    if pheno in meta.index:
        _genes = pd.Index(genes)
        print(f"{pheno}\t{100 * _genes.isin(_observed).mean():.2f}\t{100 * _genes.isin(pkn_graph).mean():.2f}")


markers in PKN: 95.45%
Cell	pObserved	pPKN
RPC	100.00	100.00
NB1	80.00	100.00
NB2	66.67	66.67
RGC	75.00	100.00
AC	50.00	100.00
Cones	75.00	100.00


In [10]:
import bonesis


In [11]:
data_meta = df_to_bonesis_data(meta[[g for g in meta.columns if g in pkn_graph]])
data_meta.keys()


dict_keys(['AC', 'Cones', 'NB1', 'NB2', 'RGC', 'RPC'])

In [12]:
pkn = bonesis.domains.InfluenceGraph(
    pkn_graph, maxclause=8, allow_skipping_nodes=True, canonic=False
)
if UNSOURCE:
    _sources = pkn.sources()
    pkn.unsource()
    print(f"unsource: positive self-loop added on {len(_sources)} nodes without inputs")
print(f"PKN: nodes={len(pkn)}, edges={pkn.number_of_edges()}")


unsource: positive self-loop added on 217 nodes without inputs
PKN: nodes=4976, edges=29004


In [13]:
bo = bonesis.BoNesis(pkn, data_meta)
bo


In [14]:
# prior_grn == the measured PKN itself (no larger CollecTRI core around it any more)
export_pkn_bundle(FILE_PREFIX, pkn_graph, pkn, data_meta, INCLUDE_HC_CONSTRAINT)

print(
    f"Exported pkn ({len(pkn)} nodes), data_meta ({len(data_meta)} observations) "
    f"and config for FILE_PREFIX={FILE_PREFIX!r}"
)


Exported pkn (4976 nodes), data_meta (6 observations) and config for FILE_PREFIX='results/hsc_4ds/grn/sum'


### Next step

Continue in **"6. BN Inference (BoNesis).ipynb"** — set `FILE_PREFIX` there to the same value as `FILE_PREFIX` above, to load the files exported here.